# Train sign language translation

Use a Kaggle GPU session with Internet enabled and extracted PHOENIX poses attached. Run cells in order. Only the training split is reduced by `SUBSET`; validation and test remain full.


## 1. Choose the experiment

`matrix` runs experiment groups; `select` runs a chosen encoder/method. Groups: `core`, `encoders`, `algos`, `ablations`, `reward`, `latency`, or `all`. The algorithm, ablation, and reward groups need a completed core checkpoint. The reward group sweeps its own weights.


In [ ]:
SUBSET = 0.05  # 0.05, 0.10, 0.25, 0.5, or 1.0
SCOPE = "matrix"  # matrix or select
GROUPS = "all"

SELECT_MODE = "single"  # single, encoder_allrl, or rl_allenc
ENCODER = "transformer"  # transformer, stgcn, gcn, graph_transformer, tcn, perceiver
ALGO = "scst"  # scst, ppo, mrt, raml, dpo
TAG = "run1"

POSE_DIR = "/kaggle/input/phoenix-poses"
WORK_DIR = "/kaggle/working"
REWARD_BLEU = 1.0
REWARD_REP = 0.5
REWARD_LEN = 0.0

assert SCOPE in {"matrix", "select"}
assert SUBSET in {0.05, 0.10, 0.25, 0.5, 1.0}


## 2. Install dependencies

PyTorch is provided by the Kaggle GPU runtime. Install `bert-score` separately only if you enable the optional BERTScore reward in the project config.


In [ ]:
%pip install -q sentencepiece sacrebleu pandas matplotlib tqdm


## 3. Load the repository


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/linhxm/Sign-Language-REL.git"
CODE_DIR = Path("/kaggle/working/Sign-Language-REL")
if not CODE_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True)
assert (CODE_DIR / "main.py").is_file(), f"Invalid repository: {CODE_DIR}"
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
os.environ["SLT_CODE_DIR"] = str(CODE_DIR)
print("Repository:", CODE_DIR)


## 4. Check input and prepare the script runner

Settings are passed to a fresh process so the repository config file stays reusable.


In [ ]:
pose_files = list(Path(POSE_DIR).glob("*.npz"))
assert pose_files, f"No pose files in {POSE_DIR}. Set POSE_DIR to the folder containing .npz files."
print("Pose files:", len(pose_files))
import json

def run_project(script, *args, data=None, training=None):
    """Run a script with temporary settings in a fresh Python process."""
    overrides = json.dumps({"data": data or {}, "train": training or {}})
    bootstrap = """
import json, runpy, sys
from configs.config import CFG
for section, values in json.loads(sys.argv[1]).items():
    for key, value in values.items():
        setattr(getattr(CFG, section), key, value)
sys.argv = sys.argv[2:]
runpy.run_path(sys.argv[0], run_name="__main__")
"""
    subprocess.run(
        [sys.executable, "-c", bootstrap, overrides, script, *map(str, args)],
        cwd=CODE_DIR, check=True,
    )


## 5. Train

This can take hours. Completed steps can be skipped when their outputs and completion markers remain in `WORK_DIR`. Preserve checkpoints and markers when continuing in a new session.


In [ ]:
data_settings = {"pose_cache_dir": POSE_DIR, "work_dir": WORK_DIR}
reward_settings = {
    "reward_bleu_weight": REWARD_BLEU,
    "reward_repetition_penalty": REWARD_REP,
    "reward_length_penalty": REWARD_LEN,
}
if SCOPE == "matrix":
    run_project("run_all.py", "--subset", SUBSET, "--groups", GROUPS,
                data=data_settings, training=reward_settings)
else:
    run_project("train_select.py", "--mode", SELECT_MODE,
                "--encoder", ENCODER, "--algo", ALGO, "--subset", SUBSET, "--tag", TAG,
                data=data_settings, training=reward_settings)


## 6. Review results


In [ ]:
import pandas as pd

run_project("scripts/aggregate_results.py", "--work_dir", WORK_DIR,
            "--out", str(Path(WORK_DIR) / "comparison_table"))
df = pd.read_csv(Path(WORK_DIR) / "comparison_table.csv")
df.sort_values(["subset_pct", "encoder", "method"], na_position="last")


## 7. Export metrics

This small archive contains metrics and comparison tables. Save checkpoints and `spm.model` separately if you need inference or continued training.


In [ ]:
import tarfile

work = Path(WORK_DIR)
files = list(work.glob("comparison_table.*"))
for directory in work.glob("*_subset*"):
    if directory.is_dir():
        for pattern in ("test_results.json", "*_history.json", "latency_*.json"):
            files.extend(directory.glob(pattern))
bundle = work / "results_bundle.tar.gz"
with tarfile.open(bundle, "w:gz") as archive:
    for path in sorted(set(files)):
        archive.add(path, arcname=str(path.relative_to(work)))
print("Saved:", bundle)


## 8. Optional: merge earlier sessions

Attach result bundles as Kaggle inputs. Combine runs from the same dataset with distinct run names; duplicate paths will overwrite earlier files.


In [ ]:
MERGE_DIR = Path(WORK_DIR) / "merged"
MERGE_DIR.mkdir(exist_ok=True)
bundles = sorted(Path("/kaggle/input").rglob("results_bundle*.tar.gz"))
assert bundles, "Attach at least one results bundle before running this optional step."
for bundle in bundles:
    with tarfile.open(bundle) as archive:
        archive.extractall(MERGE_DIR, filter="data")
run_project("scripts/aggregate_results.py", "--work_dir", MERGE_DIR,
            "--out", MERGE_DIR / "comparison_table")
run_project("scripts/make_report.py", "--work_dir", MERGE_DIR)
pd.read_csv(MERGE_DIR / "comparison_table.csv")
